# Machine learning pipeline for Immune cell type classification 

## Notebook 1: Exploratory Data Analysis 

In [1]:
#Major Imports
import os
import pickle
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
#Setting up relative paths and directories
RAND_SEED: int = 7 #Coz, 007, why not?!

DATA_DIR:      str = os.path.join(os.path.dirname(os.getcwd()),"Data")
RESULTS_DIR:   str = os.path.join(os.path.dirname(os.getcwd()),"Results")
PLOT_DIR:      str = os.path.join(os.path.dirname(os.getcwd()),"Plots")
MODEL_DIR:     str = os.path.join(os.path.dirname(os.getcwd()),"Models")

for d in [DATA_DIR, RESULTS_DIR, PLOT_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)

In [3]:
#Quick preview at the data
preview = pd.read_csv(
    os.path.join(DATA_DIR,"batch_corrected_expression_with_celltypes.tsv"),
    sep="\t",
    nrows=5
)
print(preview.shape)
print(preview.columns[:5].tolist())
preview.head()

(5, 2002)
['Unnamed: 0', 'PPBP', 'PTGDS', 'GNLY', 'FCER1A']


,Unnamed: 0,PPBP,PTGDS,GNLY,FCER1A,PF4,GZMB,MZB1,S100A9,C1QB,...,SLBP,SLC25A22,TMEM205,MORC2,CD1B,PPIL2,TCF19,PRKCB,SKA1,CellType
0,TCCCGATGTTAAGATG-1_7,-0.067185,-0.061965,-0.072270,-0.006314,0.040835,-0.018003,-0.010423,5.596175,-0.000241,...,1.366634,0.022373,0.315363,0.010218,-0.000288,0.169300,-0.010009,0.731021,0.004238,CD14 Mono
1,TTTCGATCACATTACG-1_1,-0.090013,0.010048,0.005502,0.006034,0.014521,-0.102739,-0.001492,3.366856,0.012824,...,0.692414,0.065046,0.572243,0.030051,-0.016936,0.191588,-0.004734,-0.164890,0.018072,CD14 Mono
2,AATGCCAGTGGACCAA-1_5,0.013690,0.004790,-0.030905,0.003231,0.000244,-0.053223,-0.015890,5.021750,-0.000076,...,0.393901,-0.004960,0.115961,0.483775,-0.006629,0.184109,-0.001498,1.300231,-0.001369,CD14 Mono
3,CTTAGGAAGCCAACAG-1_8,0.058790,-0.019925,0.656104,0.017998,-0.001272,0.084636,-0.108972,5.458511,-0.000183,...,0.893983,0.015698,-0.108487,0.109554,-0.000833,0.200299,-0.108342,0.462959,0.001720,CD14 Mono
4,GAATCACGTATTTCCT-1_13,0.005776,0.012726,-0.046718,-0.046348,0.006890,0.004545,-0.007758,5.781392,0.000009,...,0.274805,0.009353,0.527105,0.137788,-0.000670,0.159736,0.001091,1.009797,0.007636,CD14 Mono


What we are looking at:

1. 2002 total columns: "Unnamed: 0" is the cell barcode; then there are 2000 gene expression columns; and lastly the "CellType" label column.
2. On a quick glance, the feature values look like they have been appropriately scaled and normalised during the Seurat preprocessing stage. That is in line with the filename, i.e. "batch corected"

In [4]:
#Initiating the full load of the data
rna_seq_data = pd.read_csv(
    os.path.join(DATA_DIR, "batch_corrected_expression_with_celltypes.tsv"),
    sep = "\t",
    index_col=0
)

print(f"Shape of the data: {rna_seq_data.shape}")
print(f"Memory Usage in GB: {round(rna_seq_data.memory_usage(deep=True).sum()/1e9,2)}")
print(f"LAst Column: {rna_seq_data.columns[-1]}")
rna_seq_data.head()

Shape of the data: (76893, 2001)
Memory Usage in GB: 1.24
LAst Column: CellType


,PPBP,PTGDS,GNLY,FCER1A,PF4,GZMB,MZB1,S100A9,C1QB,C1QA,...,SLBP,SLC25A22,TMEM205,MORC2,CD1B,PPIL2,TCF19,PRKCB,SKA1,CellType
TCCCGATGTTAAGATG-1_7,-0.067185,-0.061965,-0.072270,-0.006314,0.040835,-0.018003,-0.010423,5.596175,-0.000241,-0.007603,...,1.366634,0.022373,0.315363,0.010218,-0.000288,0.169300,-0.010009,0.731021,0.004238,CD14 Mono
TTTCGATCACATTACG-1_1,-0.090013,0.010048,0.005502,0.006034,0.014521,-0.102739,-0.001492,3.366856,0.012824,-0.009514,...,0.692414,0.065046,0.572243,0.030051,-0.016936,0.191588,-0.004734,-0.164890,0.018072,CD14 Mono
AATGCCAGTGGACCAA-1_5,0.013690,0.004790,-0.030905,0.003231,0.000244,-0.053223,-0.015890,5.021750,-0.000076,0.334029,...,0.393901,-0.004960,0.115961,0.483775,-0.006629,0.184109,-0.001498,1.300231,-0.001369,CD14 Mono
CTTAGGAAGCCAACAG-1_8,0.058790,-0.019925,0.656104,0.017998,-0.001272,0.084636,-0.108972,5.458511,-0.000183,0.015630,...,0.893983,0.015698,-0.108487,0.109554,-0.000833,0.200299,-0.108342,0.462959,0.001720,CD14 Mono
GAATCACGTATTTCCT-1_13,0.005776,0.012726,-0.046718,-0.046348,0.006890,0.004545,-0.007758,5.781392,0.000009,-0.002512,...,0.274805,0.009353,0.527105,0.137788,-0.000670,0.159736,0.001091,1.009797,0.007636,CD14 Mono


Nexxt we move ahead with a rudimentary data sanity check on whether we have missing values, infinitie values, non numeric characters etc. in the dataset which could contaminate the downstream analysis. On top of that, we will also ensure that each of the individual cell in this gene expression matrix has a unique cell barcode. 

In [5]:
#Isolating the gene feature columns 
gene_columns = rna_seq_data.columns[:-1] #All the way upto but not including CellTypes columns

# Missing values (Nan)
n_missing = rna_seq_data.isna().sum().sum()
print(f"Total count of missing values: {n_missing}")

#Infinite values
n_infinite = np.isinf(rna_seq_data[gene_columns]).sum().sum()
print(f"Infinite values found: {n_infinite}")


#Duplicates in gene identifier barcodes
n_duplicates = rna_seq_data.index.duplicated().sum()
print(f"Duplicated index entries: {n_duplicates}")

#Non numeric 
non_numeric = rna_seq_data[gene_columns].select_dtypes(exclude=[np.number]).columns
print(f"Names of the non numeric gene columns: {list(non_numeric)}")

Total count of missing values: 0
Infinite values found: 0
Duplicated index entries: 0
Names of the non numeric gene columns: []


The above output indicates that I can now go ahead with cell type analysis and dimensionality reduciton work without worrying about the structural integrity of the data. 

## Analysing Cell types and their distribution 

In [6]:
#Calculating cell counts and % per each cell type
cell_type_df = rna_seq_data["CellType"].value_counts().ren